### Qué hago en esta parte (unión DRIVE + TRANSIT)

- Cargo los dos CSV: `combinado1.csv` (coche) y `transit.csv` (transporte público).
- “Arreglo” columnas para que coincidan bien al unir:
  - Quito espacios raros en `origen` y `destino`.
  - Pongo `hora_salida` con formato tipo `01:00`.
  - Convierto `fecha_consulta` a fecha real.
- Me quedo solo con filas DRIVE en el dataset de coche y solo TRANSIT en el dataset de público.
- Creo 2 columnas finales de duración:
  - `duracion_drive_min`: si hay `duracion_con_trafico_min` uso esa, si no uso `duracion_sin_trafico_min`.
  - `duracion_transit_min`: uso `duracion_sin_trafico_min` porque en TRANSIT la de tráfico suele venir vacía.
- Hago el `merge` por `origen, destino, fecha_consulta, hora_salida` para comparar el mismo trayecto en el mismo momento.
- Creo `diferencia_min` y `ratio`, que son las variables que se analizan en el proyecto.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

drive = pd.read_csv("combinado1.csv")
transit = pd.read_csv("transit.csv")

# Normalizar para que el merge no falle por espacios/tipos
for df in (drive, transit):
    df["origen"] = df["origen"].astype(str).str.strip()
    df["destino"] = df["destino"].astype(str).str.strip()
    df["hora_salida"] = df["hora_salida"].astype(str).str.zfill(5)
    df["fecha_consulta"] = pd.to_datetime(df["fecha_consulta"])

# Filtrar por modo
drive = drive[drive["modo_transporte"] == "DRIVE"].copy()
transit = transit[transit["modo_transporte"] == "TRANSIT"].copy()

# Duración principal de cada uno
# DRIVE: usa con_trafico cuando exista, si no sin_trafico
drive["duracion_drive_min"] = np.where(
    drive["duracion_con_trafico_min"].notna(),
    drive["duracion_con_trafico_min"],
    drive["duracion_sin_trafico_min"]
)

# TRANSIT: usamos la sin_trafico (la con_trafico es NaN)
transit["duracion_transit_min"] = transit["duracion_sin_trafico_min"]

# Merge emparejado por trayecto + momento
df = pd.merge(
    drive[["origen","destino","hora_salida","fecha_consulta","distancia_km","duracion_drive_min"]],
    transit[["origen","destino","hora_salida","fecha_consulta","duracion_transit_min"]],
    on=["origen","destino","hora_salida","fecha_consulta"],
    how="inner"
)

print("Filas emparejadas:", len(df))

# Variables del estudio
df["diferencia_min"] = df["duracion_transit_min"] - df["duracion_drive_min"]
df["ratio"] = df["duracion_transit_min"] / df["duracion_drive_min"]

ModuleNotFoundError: No module named 'pandas'

### Qué hago en esta parte (tablas, estadística y resultados)

- Genero tablas de frecuencia por intervalos para:
  - `duracion_drive_min` (coche)
  - `duracion_transit_min` (público)
  - `diferencia_min` (público - coche)
  Estas tablas son las típicas de estadística: ni, fi, acumuladas y porcentaje.
- Saco descriptiva (`describe`) para ver medias, cuartiles, etc.
- Calculo Pearson entre coche y público (si un trayecto tarda más en coche, suele tardar más en público).
- Hago inferencial:
  - t-test emparejado (porque es el mismo trayecto comparado en dos modos).
  - IC 95% para la media de `diferencia_min`.
- Guardo todo en `output/` para meterlo en el informe sin estar copiando a mano. XDD

In [3]:
def tabla_frecuencias_intervalos(s, bins):
    cat = pd.cut(s, bins=bins, right=False, include_lowest=True)
    ni = cat.value_counts().sort_index()
    N = ni.sum()
    fi = ni / N
    return pd.DataFrame({
        "ni": ni,
        "fi": fi.round(4),
        "Ni": ni.cumsum(),
        "Fi": fi.cumsum().round(4),
        "%": (fi*100).round(2)
    })

bins_t = [0,10,20,30,45,60,90,120]
bins_d = [-60,-30,-15,0,10,20,30,60,120]

tabla_drive = tabla_frecuencias_intervalos(df["duracion_drive_min"], bins_t)
tabla_transit = tabla_frecuencias_intervalos(df["duracion_transit_min"], bins_t)
tabla_diff = tabla_frecuencias_intervalos(df["diferencia_min"], bins_d)

print("Frec DRIVE:\n", tabla_drive)
print("\nFrec TRANSIT:\n", tabla_transit)
print("\nFrec diferencia (TRANSIT-DRIVE):\n", tabla_diff)

# Guardar para meter en el PDF
tabla_drive.to_csv("tabla_frec_drive.csv")
tabla_transit.to_csv("tabla_frec_transit.csv")
tabla_diff.to_csv("tabla_frec_diferencia.csv")

Frec DRIVE:
                      ni      fi    Ni      Fi      %
duracion_drive_min                                  
[0, 10)              39  0.0349    39  0.0349   3.49
[10, 20)            462  0.4140   501  0.4489  41.40
[20, 30)            435  0.3898   936  0.8387  38.98
[30, 45)             85  0.0762  1021  0.9149   7.62
[45, 60)             24  0.0215  1045  0.9364   2.15
[60, 90)             49  0.0439  1094  0.9803   4.39
[90, 120)            22  0.0197  1116  1.0000   1.97

Frec TRANSIT:
                        ni      fi    Ni      Fi      %
duracion_transit_min                                  
[0, 10)                19  0.0160    19  0.0160   1.60
[10, 20)               95  0.0798   114  0.0958   7.98
[20, 30)              236  0.1983   350  0.2941  19.83
[30, 45)              326  0.2739   676  0.5681  27.39
[45, 60)              203  0.1706   879  0.7387  17.06
[60, 90)              174  0.1462  1053  0.8849  14.62
[90, 120)             137  0.1151  1190  1.0000  11.51

## Resumen descriptivo (`describe()`)

En esta sección se obtiene un resumen estadístico automático de las variables principales del estudio usando `describe()` en pandas.  
Las variables analizadas son:

- **`duracion_drive_min`**: duración del trayecto en coche (min).
- **`duracion_transit_min`**: duración del trayecto en transporte público (min).
- **`diferencia_min`**: diferencia de duración (**público − coche**) en minutos.

El comando `describe()` devuelve, para cada variable:

- **`count`**: número de observaciones válidas (datos usados en el cálculo).
- **`mean`**: media (promedio) de la duración.
- **`std`**: desviación estándar (cuánto se dispersan los valores respecto a la media).
- **`min`** y **`max`**: valores mínimo y máximo observados.
- **`25%`**, **`50%`**, **`75%`**: cuartiles (Q1, mediana y Q3).  
  - El **50%** es la **mediana**, que indica el valor central: la mitad de los trayectos dura menos y la otra mitad dura más.

In [4]:
df[["duracion_drive_min", "duracion_transit_min", "diferencia_min"]].describe()

,duracion_drive_min,duracion_transit_min,diferencia_min
count,1391.000000,1391.000000,1391.000000
mean,64.623652,72.673113,8.049461
std,87.336185,75.018465,54.548856
min,5.000000,9.000000,-194.600000
25%,17.200000,29.750000,0.600000
50%,23.900000,45.200000,12.300000
75%,58.600000,86.050000,33.100000
max,380.200000,561.200000,440.300000
